# 08 · Paper figures

Regenerates every manuscript figure from the stored result tables. **No
simulation runs here** — if a figure changes, it is because the analysis
changed, not because a random seed moved.

Outputs land in `results/figures/` as vector PDF plus 300-dpi PNG.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import os; os.chdir(ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

from telemetry_freshness.viz import style as S, figures as F
S.use_paper_style()
print("working dir:", ROOT)

working dir: /path/to/Field_Measurement_of_Age_of_Information


In [2]:
from pathlib import Path
RAW = Path("results/raw")
loaded = {}
for key, fn in [("E1","E1_pairwise_sweep.parquet"), ("E2","E2_idfds_downsampling.parquet"),
                ("E3","E3_maneuver_sensitivity.parquet"), ("E4","E4_density_sweep.parquet")]:
    p = RAW / fn
    if p.exists():
        loaded[key] = pd.read_parquet(p)
        print(f"{key}: {len(loaded[key]):,} rows")
    else:
        print(f"{key}: MISSING ({fn}) — run the upstream notebook first")

E1: 540,000 rows
E2: 709,050 rows
E3: 255,258 rows
E4: 272,673 rows


In [3]:
built = []

if "E2" in loaded:
    fig, _ = F.fig_aoi_vs_position_error(
        loaded["E2"], subtitle="IDF-DS · 12 real fixed-wing flights · 62 Hz reference")
    built += S.save(fig, "F1_aoi_vs_position_error"); S.plt.close(fig)

if "E1" in loaded:
    fig, _ = F.fig_ttc_error_vs_aoi(loaded["E1"], level_col="encounter_type")
    built += S.save(fig, "F3_ttc_error_vs_aoi"); S.plt.close(fig)

    fig, _ = F.fig_rate_latency_heatmap(loaded["E1"], value_col="ttc_error_abs_s", agg="p95")
    built += S.save(fig, "F5_rate_latency_heatmap"); S.plt.close(fig)

    fig, _ = F.fig_rate_vs_realised_aoi(loaded["E1"])
    built += S.save(fig, "F8_rate_vs_realised_aoi"); S.plt.close(fig)

for name, fn, kwargs in [
    ("F2_recall_vs_aoi_by_density", "results/processed/E4_recall_by_aoi_density.csv",
     dict(level_col="density_label", requirement=0.99)),
]:
    p = Path(fn)
    if p.exists():
        d = pd.read_csv(p)
        from telemetry_freshness.evaluation.statistics import wilson_interval
        lh = [wilson_interval(int(r.tp), int(r.n_positive)) for _, r in d.iterrows()]
        d["recall_lo"] = [x[0] for x in lh]; d["recall_hi"] = [x[1] for x in lh]
        fig, _ = F.fig_recall_vs_aoi(d, **kwargs)
        built += S.save(fig, name); S.plt.close(fig)

p = Path("results/processed/E1_lead_time_by_aoi.csv")
if p.exists():
    fig, _ = F.fig_lead_time_vs_aoi(pd.read_csv(p), level_col="encounter_type", min_lead_s=3.0)
    built += S.save(fig, "F4_lead_time_vs_aoi"); S.plt.close(fig)

print(f"{len(built)} files written")
for b in built: print("  ", b)

12 files written
   results/figures/F1_aoi_vs_position_error.pdf
   results/figures/F1_aoi_vs_position_error.png
   results/figures/F3_ttc_error_vs_aoi.pdf
   results/figures/F3_ttc_error_vs_aoi.png
   results/figures/F5_rate_latency_heatmap.pdf
   results/figures/F5_rate_latency_heatmap.png
   results/figures/F8_rate_vs_realised_aoi.pdf
   results/figures/F8_rate_vs_realised_aoi.png
   results/figures/F2_recall_vs_aoi_by_density.pdf
   results/figures/F2_recall_vs_aoi_by_density.png
   results/figures/F4_lead_time_vs_aoi.pdf
   results/figures/F4_lead_time_vs_aoi.png


## Manifest

In [4]:
figs = sorted(Path("results/figures").glob("*.pdf"))
manifest = pd.DataFrame({
    "figure": [f.stem for f in figs],
    "pdf_kb": [round(f.stat().st_size/1024, 1) for f in figs],
    "png_exists": [f.with_suffix(".png").exists() for f in figs],
})
display(manifest)
manifest.to_csv("results/figures/manifest.csv", index=False)

,figure,pdf_kb,png_exists
0,F0_bubbles_aoi_cdf,47.2,True
1,F1_aoi_vs_position_error,30.1,True
2,F1b_phase_position_vs_heading,40.8,True
3,F2_recall_vs_aoi_by_density,38.6,True
4,F3_ttc_error_vs_aoi,43.6,True
5,F4_lead_time_vs_aoi,35.6,True
6,F5_rate_latency_heatmap,27.4,True
7,F6_missed_conflicts_vs_density,39.1,True
8,F6b_pooled_recall_curve,35.1,True
9,F7_threshold_sensitivity,23.1,True
